# VaR Three Ways, on the DAX

**pyportfolios.com tutorial T09** · DAX (^GDAXI), Jan 2010 – Dec 2024 · NumPy · SciPy · Pandas · Polars · DuckDB

Value-at-Risk is the industry's standard downside number — daily loss limits and
regulatory capital both hang off it. In this notebook we

1. compute 99% and 95% VaR (and CVaR) three ways — historical simulation,
   parametric (normal **and** Student-t), and Monte Carlo,
2. backtest all of them out-of-sample with a rolling 250-day window and
   Kupiec's proportion-of-failures test, and
3. recompute the historical quantile with **Polars** expressions and **DuckDB**
   SQL — the modern data-engineering route to the exact same number.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats

plt.rcParams["figure.figsize"] = (10, 5)
SEED = 42

## 1 · Data: fifteen years of the DAX

The DAX (^GDAXI) from January 2010 through December 2024 — the euro crisis,
the 2015-16 China scare, COVID, the 2022 energy shock. A proper stress diet
for a risk model.

In [ ]:
px = yf.download("^GDAXI", start="2010-01-01", end="2025-01-01",
                 auto_adjust=True, progress=False)["Close"].squeeze().dropna()
ret = px.pct_change().dropna()
print(f"{len(ret)} daily returns, worst day {ret.idxmin().date()} ({ret.min():.2%})")
ret.plot(lw=0.5, title="DAX daily returns");

## 2 · Historical simulation

No model at all: VaR at level α is the empirical (1−α)-quantile of returns,
negated. CVaR (expected shortfall) is the mean return beyond that quantile.

In [ ]:
def hist_var_cvar(x, alpha=0.99):
    q = np.quantile(x, 1 - alpha)
    return -q, -x[x <= q].mean()

for a in (0.99, 0.95):
    v, c = hist_var_cvar(ret.values, a)
    print(f"historical  {a:.0%}: VaR = {v:.2%}   CVaR = {c:.2%}")

## 3 · Parametric: normal, then Student-t

The variance–covariance shortcut assumes a distribution and reads the quantile
off its formula. With a normal that famously understates the tail; refitting
the same idea with a Student-t (degrees of freedom estimated by MLE) is a
one-line upgrade that buys most of the missing tail mass.

In [ ]:
mu, sd = ret.mean(), ret.std(ddof=1)
t_df, t_loc, t_scale = stats.t.fit(ret.values)
print(f"fitted Student-t df = {t_df:.2f}")

def normal_var_cvar(alpha):
    z = stats.norm.ppf(1 - alpha)
    return -(mu + sd * z), -mu + sd * stats.norm.pdf(z) / (1 - alpha)

def t_var_cvar(alpha):
    p = 1 - alpha
    x = stats.t.ppf(p, t_df)
    var = -(t_loc + t_scale * x)
    cvar = -t_loc + t_scale * stats.t.pdf(x, t_df) * (t_df + x**2) / ((t_df - 1) * p)
    return var, cvar

for a in (0.99, 0.95):
    nv, nc = normal_var_cvar(a); tv, tc = t_var_cvar(a)
    print(f"{a:.0%}  normal: VaR {nv:.2%} CVaR {nc:.2%}   t: VaR {tv:.2%} CVaR {tc:.2%}")

## 4 · Monte Carlo

Simulate from the fitted Student-t (200,000 seeded draws) and read the
empirical tail of the simulation. On a single linear asset this must agree
with the analytic t to Monte-Carlo error — the payoff comes when the
portfolio has options or path dependence and no closed form exists.

In [ ]:
rng = np.random.default_rng(SEED)
draws = t_loc + t_scale * rng.standard_t(t_df, 200000)

for a in (0.99, 0.95):
    v, c = hist_var_cvar(draws, a)
    print(f"monte carlo {a:.0%}: VaR = {v:.2%}   CVaR = {c:.2%}")

## 5 · Backtest: rolling window + Kupiec

A VaR is a falsifiable forecast: at 99% the next-day loss should exceed it on
about 1% of days. We re-estimate each method on a rolling 250-day window
(t and MC refit every 20 days — desk practice), forecast one day
ahead, count breaches, and test the breach *rate* with Kupiec's (1995)
proportion-of-failures likelihood ratio, which is χ²(1) under H₀.

In [ ]:
WINDOW, REFIT = 250, 20
rv, n = ret.values, len(ret)
z01 = stats.norm.ppf(0.01)

f_hist = ret.rolling(WINDOW).quantile(0.01).shift(1).values
f_norm = (ret.rolling(WINDOW).mean() + ret.rolling(WINDOW).std(ddof=1) * z01).shift(1).values

f_t, f_mc = np.full(n, np.nan), np.full(n, np.nan)
for s in range(WINDOW, n, REFIT):
    dfw, locw, scalew = stats.t.fit(rv[s - WINDOW:s])
    e = min(s + REFIT, n)
    f_t[s:e] = locw + scalew * stats.t.ppf(0.01, dfw)
    f_mc[s:e] = np.quantile(locw + scalew * rng.standard_t(dfw, 200000), 0.01)

def kupiec(x, n, p=0.01):
    phat = x / n
    ll0 = (n - x) * np.log(1 - p) + x * np.log(p)
    ll1 = (n - x) * np.log(1 - phat) + (x * np.log(phat) if x > 0 else 0.0)
    lr = -2 * (ll0 - ll1)
    return lr, stats.chi2.sf(lr, 1)

n_fc = n - WINDOW
for name, f in [("hist", f_hist), ("normal", f_norm), ("t", f_t), ("mc", f_mc)]:
    valid = ~np.isnan(f)
    x = int((rv[valid] < f[valid]).sum())
    lr, p = kupiec(x, n_fc)
    print(f"{name:>6}: {x:>3} breaches / {n_fc} (expected {0.01 * n_fc:.1f})  "
          f"Kupiec LR = {lr:.2f}  p = {p:.4f}")

In [ ]:
plt.plot(ret.index[WINDOW:], rv[WINDOW:], lw=0.4, color="grey", label="daily return")
plt.plot(ret.index[WINDOW:], f_hist[WINDOW:], lw=1.4, label="rolling 99% -VaR (hist)")
breach = rv < f_hist
plt.scatter(ret.index[breach], rv[breach], s=14, color="crimson", zorder=3, label="breach")
plt.legend(); plt.title("DAX daily returns vs rolling 250d historical 99% VaR");
plt.show()

## 6 · The same VaR in Polars and DuckDB

Historical VaR is just a quantile over a column — exactly the shape of problem
modern engines eat. Polars evaluates a lazy expression pipeline over the CSV;
DuckDB runs SQL directly against the file. Both stream, so the identical code
works when "15 years of one index" becomes "10 years of every book in the
firm". The three engines must agree to floating-point noise — if they don't,
the bug is yours, not theirs.

In [ ]:
import polars as pl
import duckdb
import time

ret.rename("ret").to_frame().to_csv("dax_returns.csv", index_label="date")

t0 = time.perf_counter()
q_pd = float(np.quantile(rv, 0.01))
print(f"pandas/NumPy   {q_pd:.15f}   {(time.perf_counter() - t0) * 1e3:6.2f} ms")

t0 = time.perf_counter()
q_pl = float(
    pl.scan_csv("dax_returns.csv")
      .select(pl.col("ret").quantile(0.01, interpolation="linear"))
      .collect()
      .item()
)
print(f"polars (lazy)  {q_pl:.15f}   {(time.perf_counter() - t0) * 1e3:6.2f} ms")

t0 = time.perf_counter()
q_db = float(duckdb.sql(
    "SELECT quantile_cont(ret, 0.01) FROM read_csv('dax_returns.csv')"
).fetchone()[0])
print(f"duckdb SQL     {q_db:.15f}   {(time.perf_counter() - t0) * 1e3:6.2f} ms")

# timing note: at 3,805 rows engine startup dominates; the streaming engines
# win once the file no longer fits in memory.
print(f"max |diff| = {max(abs(q_pl - q_pd), abs(q_db - q_pd)):.2e}  (must be <= 1e-12)")

## Takeaways

- The three recipes disagree, and the disagreement is informative: at 99% the normal sits ~60bp below historical, t and Monte Carlo.
- The fitted Student-t degrees of freedom (~3) say DAX tails are far from Gaussian; the t roughly halves the normal's excess breaches.
- Honest backtest result: *every* unconditional method breaches too often at 99% — breaches cluster in vol regimes, which is Kupiec's message here and the motivation for conditional models (GARCH, filtered historical simulation).
- CVaR comes almost free alongside every method and is the better number for limits.
- A quantile is a quantile: pandas, Polars and DuckDB agree to 1e-15 — pick the engine for the data size, not the answer.
- Never ship a VaR you have not backtested.

*© pyportfolios.com — runnable companion to the article. Data: Yahoo Finance via yfinance.*